In [ ]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

In [ ]:
# 2. Get a fresh copy of main
%cd /content
!rm -rf kv-evidence-loss
!git clone -q https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
# 3. Install the pinned team environment (torch is Colab's own CUDA build)
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

In [ ]:
# 4. Record the full environment of this run (GPU, driver, Python, CUDA, every package); one run writes both budgets
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; pip freeze) > results/answers/icr_0.2_env.txt
!cp results/answers/icr_0.2_env.txt results/answers/icr_0.1_env.txt
!head -4 results/answers/icr_0.2_env.txt

In [ ]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

In [ ]:
# 6. Test on the first 2 questions (first run downloads Qwen3-8B, ~16 GB); writes *_limit2 files and results/raw/kept_limit2/
!python scripts/icr.py --limit 2

In [ ]:
# 7. Budget 1.0 check: every paragraph kept, so the answers must equal the full-cache baseline
!python scripts/icr.py --budgets 1.0

In [ ]:
# 8. Compare with full_1.0 (some HotpotQA questions have fewer than 10 paragraphs, so the count is per question)
import pandas as pd
icr_full = pd.read_parquet("results/answers/icr_1.0.parquet")
full = pd.read_parquet("results/answers/full_1.0.parquet")
n_paragraphs = pd.read_parquet("results/icr_scores.parquet").groupby(level=0).size()
assert icr_full["n_paragraphs_kept"].eq(n_paragraphs.loc[icr_full.index]).all() and icr_full["prompt_kept_fraction"].eq(1.0).all()
same = icr_full["answer"].eq(full.loc[icr_full.index, "answer"])
print(f"answers equal to full_1.0: {same.sum()}/{len(same)}")
print(pd.concat([full.loc[~same, "answer"], icr_full.loc[~same, "answer"]], axis=1, keys=["full", "icr_1.0"]))
# the check's masks are not part of the results; remove them so that retention.py does not read them
!rm -rf results/raw/kept/icr/1.0

In [ ]:
# 9. Full run: 169 questions, budgets 0.2 and 0.1 (one ranking per question, shared by both budgets)
!python scripts/icr.py

In [ ]:
# 10. Look at the results: A/B kept rates, accuracy (all and model-filter keep), ranks of A and B
pd.set_option("display.max_colwidth", 60)
keep = pd.read_parquet("data/hotpotqa/filter_qwen3_8b.parquet")["keep"]
rows = {}
for b in ["0.2", "0.1"]:
    df = pd.read_parquet(f"results/answers/icr_{b}.parquet")
    k = df[keep.loc[df.index]]
    rows[b] = {"contains_alias": df["contains_alias"].mean(), "contains_alias_keep": k["contains_alias"].mean(), "f1": df["f1"].mean(),
               "a_kept": df["a_kept"].mean(), "b_kept": df["b_kept"].mean(), "both_kept": (df["a_kept"] & df["b_kept"]).mean(),
               "paragraphs": df["n_paragraphs_kept"].mean(), "prompt_kept_fraction": df["prompt_kept_fraction"].mean()}
rows["full"] = {"contains_alias": full["contains_alias"].mean(), "contains_alias_keep": full.loc[keep[keep].index, "contains_alias"].mean(), "f1": full["f1"].mean()}
print(pd.DataFrame(rows).T.round(3))
scores = pd.read_parquet("results/icr_scores.parquet")
print(pd.DataFrame({"A": scores[scores["is_a"]]["rank"].value_counts().sort_index(), "B": scores[scores["is_b"]]["rank"].value_counts().sort_index()}).fillna(0).astype(int))

In [ ]:
# 11. Copy the masks to Google Drive (results/raw/ is not in git)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/kv-evidence-loss/raw/kept
!cp -r results/raw/kept/icr /content/drive/MyDrive/kv-evidence-loss/raw/kept/
!ls /content/drive/MyDrive/kv-evidence-loss/raw/kept/icr/*/ | head

In [ ]:
# 12. Download the answers, configs, scores and environment records
from google.colab import files
for b in ["0.2", "0.1"]:
    files.download(f"results/answers/icr_{b}.parquet")
    files.download(f"results/answers/icr_{b}_config.json")
    files.download(f"results/answers/icr_{b}_env.txt")
files.download("results/icr_scores.parquet")